In [ ]:
import numpy as np 
import pandas as pd 
from datetime import datetime as dt
from collections import Counter

import seaborn as sns
import matplotlib.pyplot as plt
import missingno as msno


import warnings
from warnings import filterwarnings



from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, cross_validate, RandomizedSearchCV, cross_val_score, train_test_split
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

from sklearn.exceptions import DataConversionWarning


pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)
pd.set_option('display.width', 500)
warnings.filterwarnings("ignore")
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=DeprecationWarning)
warnings.simplefilter(action='ignore', category=DataConversionWarning)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
train = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")

#check the numbers of samples and features
print("The train data size before dropping Id feature is : {} ".format(train.shape))
print("The test data size before dropping Id feature is : {} ".format(test.shape))

#Save the 'Id' column
train_ID = train['Id']
test_ID = test['Id']

#Now drop the  'Id' colum since it's unnecessary for  the prediction process.
train.drop("Id", axis = 1, inplace = True)
test.drop("Id", axis = 1, inplace = True)

#check again the data size after dropping the 'Id' variable
print("\nThe train data size after dropping Id feature is : {} ".format(train.shape)) 
print("The test data size after dropping Id feature is : {} ".format(test.shape))

# I will concatenate the two dataframes
#df = pd.concat([train,test], axis=0)

ntrain = train.shape[0]
ntest = test.shape[0]
y_train = train.SalePrice.values
df = pd.concat((train, test)).reset_index(drop=True)
df.drop(['SalePrice'], axis=1, inplace=True)
print("df size is : {}".format(df.shape))

In [ ]:
train.head()

In [ ]:
# train["SalePrice"].isnull().sum()

In [ ]:
# check_outlier(train, "SalePrice")

In [ ]:
test.head(10)

In [ ]:
df.tail()

In [ ]:
def check_df(dataframe, head=5):
    print('\033[1m' + 10*"*" + ' SHAPE ' + 10*"*" + '\033[0m')
    print(f"Rows:{dataframe.shape[0]}")
    print(f"Columns:{dataframe.shape[1]}")
    print('\033[1m' + 10*"*" + ' TYPES ' + 10*"*" + '\033[0m')
    print(dataframe.dtypes)
    print('\033[1m' + 10*"*" + ' NUNIQUE ELEMENTS ' + 10*"*" + '\033[0m')
    print(dataframe.nunique())
    print('\033[1m' + 10*"*" + ' NA ' + 10*"*" + '\033[0m')
    print(dataframe.isnull().sum())
    print('\033[1m' + 10*"*" + ' DESCRIBE ' + 10*"*" + '\033[0m')
    print(dataframe.describe().T)
    print('\033[1m' + 10*"*" + ' DUPLICATED VALUE ' + 10*"*" + '\033[0m')
    print(dataframe.duplicated().sum())
    print('\033[1m' + 10*"*" + ' HEAD ' + 10*"*" + '\033[0m')
    print(dataframe.head(head))
    
check_df(df)

In [ ]:
def grab_col_names(dataframe, cat_th=10, car_th=20):
    
    """
    It gives the names of categorical, numerical and categorical but cardinal variables in the data set.
    Note: Categorical variables with numerical appearance are also included in categorical variables.

    Parameters
    ------
            df: Dataframe
                The dataframe from which variable names are to be retrieved
        cat_th: int, optional
                threshold value for numeric but categorical variables
        car_th: int, optinal
                threshold value for categorical but cardinal variables

    Returns
    ------
        cat_cols: list
                Categorical variable list
        num_cols: list
                Numeric variable list
        cat_but_car: list
                Categorical but cardinal variable list

    Notes
    ------
        cat_cols + num_cols + cat_but_car = total number of variables
        num_but_cat is inside cat_cols
   """
    cat_cols = [col for col in dataframe.columns if str(dataframe[col].dtypes) in ["category", "object", "bool"]]
    num_but_cat = [col for col in dataframe.columns if (dataframe[col].nunique() < 10) and dataframe[col].dtypes in ["int64", "float64"]]
    cat_but_car = [col for col in dataframe.columns if
                 (dataframe[col].nunique() > 20) and str(dataframe[col].dtypes) in ["category", "object"]]
    cat_cols = cat_cols + num_but_cat
    cat_cols = [col for col in cat_cols if col not in cat_but_car]

    num_cols = [col for col in dataframe.columns if dataframe[col].dtypes in ["int64", "float64"]]
    num_cols = [col for col in num_cols if col not in cat_cols]
    
    print(f"Observations: {dataframe.shape[0]}")
    print(f"Variables: {dataframe.shape[1]}")
    print(f"cat_cols: {len(cat_cols)}")
    print(f"num_cols: {len(num_cols)}")
    print(f"cat_but_car : {len(cat_but_car)}")
    print(f"num_but_cat : {len(num_but_cat)}")

    return cat_cols,num_cols,cat_but_car


In [ ]:
cat_cols,num_cols,cat_but_car= grab_col_names(df)

In [ ]:
num_but_cat = [col for col in df.columns if (df[col].nunique() < 10) and df[col].dtypes in ["int64", "float64"]]

In [ ]:
num_but_cat

In [ ]:
cat_cols

In [ ]:
num_cols

In [ ]:
def num_summary(dataframe, numerical_col, plot=False):
    quantiles = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 0.99]
    print(dataframe[numerical_col].describe(quantiles).T)

    if plot:
        dataframe[numerical_col].hist()
        plt.xlabel(numerical_col)
        plt.title(numerical_col)
        plt.show(block=True)


for col in df.columns:
    if df[col].dtypes == "bool":
        df[col] = df[col].astype(int)

In [ ]:
for col in num_cols:
    num_summary(df, col, plot=True)

In [ ]:
def cat_summary(dataframe, col_name, plot=False):
    print(pd.DataFrame({col_name: dataframe[col_name].value_counts(),
                        "Ratio": 100 * dataframe[col_name].value_counts() / len(dataframe)}))
    print("##########################################")

    if plot:
        sns.countplot(x=dataframe[col_name], data=dataframe)
        plt.show(block=True)

In [ ]:
for col in cat_cols:
    cat_summary(df, col, plot=True)

In [ ]:
# def target_summary_with_cat(dataframe, target, categorical_col):
#     print(pd.DataFrame({"TARGET_MEAN": dataframe.groupby(categorical_col)[target].mean()}), end="\n\n\n")


# for col in cat_cols:
#     target_summary_with_cat(df, "SalePrice", col)

### Target feature visualization

In [ ]:
sns.distplot(train['SalePrice']);

In [ ]:
corrmat = train.corr()
f, ax = plt.subplots(figsize=(16, 16))
sns.heatmap(corrmat, vmax=.8, square=True)
plt.show()

### Target features correlation matrix (zoomed heatmap style)

In [ ]:
plt.figure(figsize=(16,16))
columns = corrmat.nlargest(10, 'SalePrice')['SalePrice'].index
correlation_matrix = np.corrcoef(train[columns].values.T)
sns.set(font_scale=1.25)
heat_map = sns.heatmap(correlation_matrix, cbar=True, annot=True, square=True, fmt='.2f', annot_kws={'size': 10}, yticklabels=columns.values, xticklabels=columns.values)
plt.show()

In [ ]:
figure, ax = plt.subplots(1,3, figsize = (20,8))
sns.stripplot(data=train, x = 'OverallQual', y='SalePrice', ax = ax[0])
sns.violinplot(data=train, x = 'OverallQual', y='SalePrice', ax = ax[1])
sns.boxplot(data=train, x = 'OverallQual', y='SalePrice', ax = ax[2])
plt.show()

In [ ]:
#scatter plot grlivarea/saleprice
train.plot.scatter(x='GrLivArea',y='SalePrice', ylim=(0,800000));

In [ ]:
# YearBuilt vs SalePrice

Pearson_YrBlt = 0.56
plt.figure(figsize = (12,6))
sns.regplot(data=train, x = 'YearBuilt', y='SalePrice', scatter_kws={'alpha':0.2})
plt.title('YearBuilt vs SalePrice', fontsize = 12)
plt.legend(['$Pearson=$ {:.2f}'.format(Pearson_YrBlt)], loc = 'best')
plt.show()

## Missing Values

In [ ]:
def missing_values_table(dataframe, na_name=False):
    na_columns = [col for col in dataframe.columns if dataframe[col].isnull().sum() > 0]

    n_miss = dataframe[na_columns].isnull().sum().sort_values(ascending=False)
    ratio = (dataframe[na_columns].isnull().sum() / dataframe.shape[0] * 100).sort_values(ascending=False)
    missing_df = pd.concat([n_miss, np.round(ratio, 2)], axis=1, keys=['n_miss', 'ratio'])
    print(missing_df, end="\n")

    if na_name:
        return na_columns


missing_values_table(df)

In [ ]:
sns.set_style("white")
f, ax = plt.subplots(figsize=(15, 6))
missing = round(df.isnull().mean()*100,2)
missing = missing[missing > 0]
missing.sort_values(inplace=True)
missing.plot.bar(color="#3f72af")


ax.xaxis.grid(False)
ax.set(ylabel="Percent of missing values")
ax.set(xlabel="Features")
ax.set(title="Percent missing data by feature")
sns.despine()

### Filling missing values of categorial columns

In [ ]:
df.shape

In [ ]:
# I can drop PoolQC, MiscFeature, Alley and Fence features because they have more than 80% of missing values.
df = df.drop(['Alley','PoolQC','Fence','MiscFeature'],axis=1)

In [ ]:
df.shape

In [ ]:
columns_None = ['BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2','GarageType','GarageFinish','GarageQual','FireplaceQu','GarageCond']
df[columns_None]= df[columns_None].fillna("None")

In [ ]:
df[columns_None].tail()

In [ ]:
columns_with_lowNA = ['MSZoning','Utilities','Exterior1st','Exterior2nd','MasVnrType','Electrical','KitchenQual','Functional','SaleType']

#fill missing values for each column (using its own most frequent value)
df[columns_with_lowNA] = df[columns_with_lowNA].fillna(df.mode().iloc[0])

### Filling missing values of numerical columns

In [ ]:
print((df['YrSold']-df['YearBuilt']).median())
print(df["LotFrontage"].median())

In [ ]:
#So I will fill the year with 1979 and the Lot frontage with 68
df['GarageYrBlt'] = df['GarageYrBlt'].fillna(df['YrSold']-35)
df['LotFrontage'] = df['LotFrontage'].fillna(68)

#Fill the rest of columns with 0
df = df.fillna(0)

In [ ]:
missing_values_table(df)

## Outliers

In [ ]:
num_cols = df.select_dtypes(exclude=['object']).columns

def outlier_thresholds(dataframe, col_name, q1=0.5, q3=0.95):
    quartile1 = dataframe[col_name].quantile(q1)
    quartile3 = dataframe[col_name].quantile(q3)
    interquantile_range = quartile3 - quartile1
    up_limit = quartile3 + 1.5 * interquantile_range
    low_limit = quartile1 - 1.5 * interquantile_range
    return low_limit, up_limit

# This function job's is if I have outlier it returns True
def check_outlier(dataframe, col_name):
    low_limit, up_limit = outlier_thresholds(dataframe, col_name)
    if dataframe[(dataframe[col_name] > up_limit) | (dataframe[col_name] < low_limit)].any(axis=None):
        return True
    else:
        return False
for col in num_cols:
    print(col, check_outlier(df, col))

In [ ]:
# This function's job is to change the outlier values to up and low limit value of outlier_thresholds() function.
def replace_with_thresholds(dataframe, variable):
    low_limit, up_limit = outlier_thresholds(dataframe, variable)
    dataframe.loc[(dataframe[variable] < low_limit), variable] = low_limit
    dataframe.loc[(dataframe[variable] > up_limit), variable] = up_limit

In [ ]:
for col in num_cols:
    replace_with_thresholds(df, col)

In [ ]:
for col in num_cols:
    print(col, check_outlier(df, col))

## Creating New Feature

In [ ]:
# df['Age_House']= (df['YrSold']-df['YearBuilt'])
# df['Age_House'].describe()

In [ ]:
# df[df["Age_House"]<0]

In [ ]:
# df.loc[df['YrSold'] < df['YearBuilt'],'YrSold' ] = 2009
# df['Age_House']= (df['YrSold']-df['YearBuilt'])
# df['Age_House'].describe()

In [ ]:
# #MSSubClass=The building class
# df['MSSubClass'] = df['MSSubClass'].apply(str)

# #Changing OverallCond into a categorical variable
# #df['OverallCond'] = df['OverallCond'].astype(str)
# #df["OverallQual"] = df["OverallQual"].astype(str)

# #Year and month sold are transformed into categorical features.
# df['YrSold '] = df['YrSold'].astype(str)
# df['MoSold'] = df['MoSold'].astype(str)

In [ ]:
# df["SqFtPerRoom"] = df["GrLivArea"] / (df["TotRmsAbvGrd"] +
#                                                        df["FullBath"] +
#                                                        df["HalfBath"] +
#                                                        df["KitchenAbvGr"])
# #df['OverallCond'] = df['OverallCond']
# #df['Total_Home_Quality'] = df['OverallQual'] + df['OverallCond']

# df['Total_Bathrooms'] = (df['FullBath'] + (0.5 * df['HalfBath']) +
#                                df['BsmtFullBath'] + (0.5 * df['BsmtHalfBath']))

# df["HighQualSF"] = df["1stFlrSF"] + df["2ndFlrSF"]

In [ ]:
# df.head()

In [ ]:
# bin_map  = {'TA':2,'Gd':3, 'Fa':1,'Ex':4,'Po':1,'None':0,'Y':1,'N':0,'Reg':3,'IR1':2,'IR2':1,'IR3':0,"None" : 0,
#             "No" : 2, "Mn" : 2, "Av": 3,"Gd" : 4,"Unf" : 1, "LwQ": 2, "Rec" : 3,"BLQ" : 4, "ALQ" : 5, "GLQ" : 6
#             }
# df['ExterQual'] = df['ExterQual'].map(bin_map)
# df['ExterCond'] = df['ExterCond'].map(bin_map)
# df['BsmtCond'] = df['BsmtCond'].map(bin_map)
# df['BsmtQual'] = df['BsmtQual'].map(bin_map)
# df['HeatingQC'] = df['HeatingQC'].map(bin_map)
# df['KitchenQual'] = df['KitchenQual'].map(bin_map)
# df['FireplaceQu'] = df['FireplaceQu'].map(bin_map)
# df['GarageQual'] = df['GarageQual'].map(bin_map)
# df['GarageCond'] = df['GarageCond'].map(bin_map)
# df['CentralAir'] = df['CentralAir'].map(bin_map)
# df['LotShape'] = df['LotShape'].map(bin_map)
# df['BsmtExposure'] = df['BsmtExposure'].map(bin_map)
# df['BsmtFinType1'] = df['BsmtFinType1'].map(bin_map)
# df['BsmtFinType2'] = df['BsmtFinType2'].map(bin_map)

# PavedDrive =   {"N" : 0, "P" : 1, "Y" : 2}
# df['PavedDrive'] = df['PavedDrive'].map(PavedDrive)

## Feature Scaling

In [ ]:
cat_cols,num_cols,cat_but_car= grab_col_names(df)

In [ ]:
num_cols

In [ ]:
df.head()

In [ ]:
rs = RobustScaler()
df[num_cols] = rs.fit_transform(df[num_cols])

In [ ]:
df.head()

## Encoding

In [ ]:
# Rare Encoding
def rare_encoder(dataframe, rare_perc):
    temp_df = dataframe.copy()

    rare_columns = [col for col in temp_df.columns if temp_df[col].dtypes == 'O'
                    and (temp_df[col].value_counts() / len(temp_df) < rare_perc).any(axis=None)]
    print(rare_columns)
    for var in rare_columns:
        tmp = temp_df[var].value_counts() / len(temp_df)
        rare_labels = tmp[tmp < rare_perc].index
        temp_df[var] = np.where(temp_df[var].isin(rare_labels), 'Rare', temp_df[var])

    return temp_df
df = rare_encoder(df, 0.01)

df['Street'].value_counts()

In [ ]:
df.head()

In [ ]:
 # df["Utilities"].value_counts()

In [ ]:
df.shape

In [ ]:
def label_encoder(dataframe, binary_col):
    labelencoder = LabelEncoder()
    dataframe[binary_col] = labelencoder.fit_transform(dataframe[binary_col])
    return dataframe
binary_cols = [col for col in df.columns if df[col].dtype not in ["int64", "float64"] and df[col].nunique() == 2]
print(binary_cols)
for col in binary_cols:
    label_encoder(df, col)

In [ ]:
df.shape

In [ ]:
# df["Street"].value_counts()

In [ ]:
 # df["CentralAir"].value_counts()

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
def one_hot_encoder(dataframe, categorical_cols, drop_first=True):
    dataframe = pd.get_dummies(dataframe, columns=categorical_cols, drop_first=drop_first)
    return dataframe


ohe_cols = [col for col in df.columns if 10 >= df[col].nunique() > 2]
print(ohe_cols)

df = one_hot_encoder(df, ohe_cols)

df.head()

In [ ]:
df.shape

In [ ]:
useless_cols = [col for col in df.columns if df[col].nunique() == 2 and
                (df[col].value_counts() / len(df) < 0.01).any(axis=None)]

# useless_cols = [col for col in useless_cols if col not in ['Id', 'SalePrice']]

df.drop(useless_cols, axis=1, inplace=True)

print(useless_cols)

In [ ]:
df.shape

In [ ]:
# X.drop(["Neighborhood","Exterior1st", "Exterior2nd"], axis=1, inplace=True)
# # #Train-processed de sildiğim için test-processete de düşüyorum.
# test_processed.drop(["Neighborhood","Exterior1st", "Exterior2nd"], axis=1, inplace=True)

In [ ]:
#submission için
# preds_sub = lgbm_final.predict(test_processed)
# sub = pd.DataFrame()
# sub['Id'] = test_ID
# sub['SalePrice'] = preds_sub
# sub.to_csv('submission-log.csv',index=False)

## Modelling

In [ ]:
train_processed = df[:ntrain]
test_processed = df[ntrain:]
print("Train_processed",train_processed.shape)
print("Test_processed",test_processed.shape)

In [ ]:
X = train_processed
y = train["SalePrice"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=45)
print("X_train",X_train.shape)
print("y_train",y_train.shape)
print("X_test",X_test.shape)
print("y_test",y_test.shape)

## Modelling Without Hyperparameter Optimization

In [ ]:
# models = [('LR', LinearRegression()),
#           ('RF', RandomForestRegressor()),
#           ("XGBoost", XGBRegressor(random_state=17)),
#           ("LightGBM", LGBMRegressor())]

In [ ]:
# rmse = np.mean(np.sqrt(-cross_val_score(LinearRegression(), X, y, cv=5, scoring="neg_mean_squared_error")))
# print (rmse)
# rmse_lr = np.sqrt(mean_squared_error(y_test, preds_rf))

In [ ]:
# lr= LinearRegression().fit(X,y)


In [ ]:
# cross_val(lr)

## Hyperparameter Optimization with RandomSearchCV

In [ ]:
def cross_val(model):
    pred = cross_val_score(model, X, y, cv=5)
    return pred.mean()


def plot_importance(model, features, num=len(X), save=False):
    feature_imp = pd.DataFrame({'Value': model.feature_importances_, 'Feature': features.columns})
    plt.figure(figsize=(10, 10))
    sns.set(font_scale=1)
    sns.barplot(x="Value", y="Feature", data=feature_imp.sort_values(by="Value",
                                                                     ascending=False)[0:num])
    plt.title('Features')
    plt.tight_layout()
    plt.show()
    if save:
        plt.savefig('importances.png')     

In [ ]:
#Random Forest Regressor
rf = RandomForestRegressor(random_state=17)

rf_random_params = {"max_depth": np.random.randint(5, 50, 10),
                    "max_features": [3, 5, 7, "auto", "sqrt"],
                    "min_samples_split": np.random.randint(2, 50, 20),
                    "n_estimators": [int(x) for x in np.linspace(start=200, stop=1500, num=10)]}

rf_random = RandomizedSearchCV(estimator=rf,
                               param_distributions=rf_random_params,
                               n_iter=100,  
                               cv=3,
                               verbose=True,
                               random_state=42,
                               n_jobs=-1)

rf_random.fit(X_train, y_train)

In [ ]:
print(rf_random.best_params_)

rf_random_final = rf.set_params(**rf_random.best_params_, random_state=17).fit(X, y)

In [ ]:
preds_rf = rf_random_final.predict(X_test)
mae_rf = mean_absolute_error(y_test, preds_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, preds_rf))
cv_rf = cross_val(rf_random_final)

print(mae_rf)
print(rmse_rf)
print(cv_rf)

In [ ]:
plot_importance(rf_random_final, X_train, num=10)

In [ ]:
#XGBoost Regressor
xgb = XGBRegressor(random_state=17)

xgb_params = {"learning_rate": [0.1, 0.01],
                  "max_depth": [5, 8],
                  "n_estimators": [100, 500, 1000],
                  "subsample": [0.7, 1]}

xgb_random = RandomizedSearchCV(estimator=xgb,
                               param_distributions=xgb_params, 
                               verbose=True,n_iter=100,
                               n_jobs =-1)

xgb_random.fit(X_train, y_train)

In [ ]:
print(xgb_random.best_params_)

xgb_random_final = xgb.set_params(**xgb_random.best_params_, random_state=17).fit(X, y)

In [ ]:
preds_xgb = xgb_random_final.predict(X_test)
mae_xgb = mean_absolute_error(y_test, preds_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, preds_xgb))
cv_xgb = cross_val(xgb_random_final)

print(mae_xgb)
print(rmse_xgb)
print(cv_xgb)

In [ ]:
plot_importance(xgb_random_final, X_train, num=10)

In [ ]:
#LightGBM Regressor
lgbm = LGBMRegressor(random_state=17)
lgbm_params = {"learning_rate": [0.01, 0.1],
               "n_estimators": [100, 300, 500, 1000],
               "colsample_bytree": [0.5, 0.7, 1]}

lgbm_random = RandomizedSearchCV(estimator=lgbm,
                               param_distributions=lgbm_params, 
                               verbose=True,n_iter=100,
                               n_jobs=-1)

lgbm_random.fit(X_train, y_train)

In [ ]:
print(lgbm_random.best_params_)

lgbm_random_final = lgbm.set_params(**lgbm_random.best_params_, random_state=17).fit(X, y)

In [ ]:
preds_lgbm = lgbm_random_final.predict(X_test)
mae_lgbm = mean_absolute_error(y_test, preds_lgbm)
rmse_lgbm = np.sqrt(mean_squared_error(y_test, preds_lgbm))
cv_lgbm = cross_val(lgbm_random_final)

print(mae_lgbm)
print(rmse_lgbm)
print(cv_lgbm)

In [ ]:
plot_importance(lgbm_random_final, X_train, num=10)

In [ ]:
# CatBoost Regressor
cb = CatBoostRegressor(loss_function='RMSE', logging_level='Silent')
cb_params = {"iterations": [200, 500],
                   "learning_rate": [0.01, 0.1],
                   "depth": [3, 6]}
cb_random = GridSearchCV(estimator=cb,param_grid=cb_params, n_jobs=-1)

cb_random.fit(X_train, y_train)

In [ ]:
print(cb_random.best_params_)

cb_random_final = cb.set_params(**cb_random.best_params_, random_state=17).fit(X, y)

In [ ]:
preds_cb= cb_random_final.predict(X_test)
mae_cb = mean_absolute_error(y_test, preds_cb)
rmse_cb = np.sqrt(mean_squared_error(y_test, preds_cb))
cv_cb = cross_val(cb_random_final)

print(mae_cb)
print(rmse_cb)
print(cv_cb)

In [ ]:
plot_importance(cb_random_final, X_train, num=10)

In [ ]:
# # Preforming a Random Grid Search to find the best combination of parameters

# grid = {'iterations': [1000,6000],
#         'learning_rate': [0.05, 0.005, 0.0005],
#         'depth': [4, 6, 10],
#         'l2_leaf_reg': [1, 3, 5, 9]}

# final_model = CatBoostRegressor()
# randomized_search_result = final_model.randomized_search(grid,
#                                                    X = X_train,
#                                                    y= y_train,
#                                                    verbose = False,
#                                                    plot=True)

In [ ]:
# # Final Cat-Boost Regressor

# params = {'iterations': 6000,
#           'learning_rate': 0.005,
#           'depth': 4,
#           'l2_leaf_reg': 1,
#           'eval_metric':'RMSE',
#           'early_stopping_rounds': 200,
#           'verbose': 200,
#           'random_seed': 42}
         
# cat_f = CatBoostRegressor(**params)
# cat_model_f = cat_f.fit(X_train,y_train,
#                      eval_set = (X_val,y_val),
#                      plot=True,
#                      verbose = False)

# catf_pred = cat_model_f.predict(X_val)
# catf_score = rmse(y_val, catf_pred)

In [ ]:
model_performances = pd.DataFrame({
    "Model" : ["RandomForest","XGBoost", "LGBM", "CatBoost"],
    "CV(5)" : [str(cv_rf)[0:5],str(cv_xgb)[0:5], str(cv_lgbm)[0:5], str(cv_cb)[0:5]],
    "MAE" : [str(mae_rf)[0:5],str(mae_xgb)[0:5], str(mae_lgbm)[0:5], str(mae_cb)[0:5]],
    "RMSE" : [str(rmse_rf)[0:5],str(rmse_xgb)[0:5], str(rmse_lgbm)[0:5], str(rmse_cb)[0:5]]
})

print("Sorted by :CV")
print(model_performances.sort_values(by="CV(5)", ascending=False))

In [ ]:
str(mae_xgb)[0:5]

In [ ]:
predict1 = lgbm_random_final.predict(test_processed)
predict2 = xgb_random_final.predict(test_processed)
predict3 = cb_random_final.predict(test_processed)
predict_y = ( predict1*0.40 + predict2 * 0.30 + predict3 * 0.30)

sub = pd.DataFrame({'Id': test_ID,
                           'SalePrice': predict_y})
sub.to_csv("../../kaggle/working/submission.csv", index=False)

In [ ]:
#submission için
# preds_sub = lgbm_final.predict(test_processed)
# sub = pd.DataFrame()
# sub['Id'] = test_ID
# sub['SalePrice'] = preds_sub
# sub.to_csv('submission-log.csv',index=False)